## Data Science Salaries Analysis
In this section, we load the dataset, normalize salaries, aggregate compensation by experience levels, and perform PCA dimensionality reduction.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA

# Load the dataset
salaries_df = pd.read_csv('/content/MyData/datascience_salaries.csv')
display(salaries_df.head())

In [ ]:
# 1. Normalize 'salary' column using Min-Max normalization
scaler = MinMaxScaler()
salaries_df['salary_normalized'] = scaler.fit_transform(salaries_df[['salary']])

# Verify normalization
display(salaries_df[['salary', 'salary_normalized']].describe())

In [ ]:
# 2. Group by 'experience_level' and calculate average and median salary
salary_stats = salaries_df.groupby('experience_level')['salary'].agg(['mean', 'median']).reset_index()
print("Salary statistics by experience level:")
display(salary_stats)

In [ ]:
# 3. Dimensionality Reduction using PCA
# Let's select categorical columns to encode first to prepare a complete feature matrix for PCA
categorical_cols = ['experience_level', 'employment_type', 'job_title', 'salary_currency', 'employee_residence', 'company_location', 'company_size']
# Keep only existing categorical columns in the dataframe
categorical_cols = [col for col in categorical_cols if col in salaries_df.columns]

# Prepare numerical features (including normalized salary) and dummy encoded categorical columns
numeric_cols = salaries_df.select_dtypes(include=[np.number]).columns.tolist()
if 'salary' in numeric_cols:
    numeric_cols.remove('salary') # Remove non-scaled raw salary to avoid duplication

features_df = pd.get_dummies(salaries_df[categorical_cols + numeric_cols], columns=categorical_cols, drop_first=True)

# Fill any missing values with 0 before PCA
features_df = features_df.fillna(0)

# Apply PCA to reduce the components to 2 for easier visualization/management
pca = PCA(n_components=2)
pca_results = pca.fit_transform(features_df)

# Save PCA components to the DataFrame
salaries_df['PCA1'] = pca_results[:, 0]
salaries_df['PCA2'] = pca_results[:, 1]

print("Explained variance ratio:", pca.explained_variance_ratio_)
display(salaries_df[['PCA1', 'PCA2']].head())